In [1]:
import pandas as pd
import numpy as np
import time
import yaml

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV

from sklearn.neighbors import KNeighborsRegressor

from sklearn.svm import SVR, LinearSVR

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


In [2]:
with open("d:/HousePricing/para_config.yml", "r") as f:
    config = yaml.safe_load(f)
    
print("Configuration successfully loaded from para_config.yml")

Configuration successfully loaded from para_config.yml


In [3]:
df = pd.read_csv(config["data"]["processed_data"])
df.head(10)

,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,...,zipcode_98146,zipcode_98148,zipcode_98155,zipcode_98166,zipcode_98168,zipcode_98177,zipcode_98178,zipcode_98188,zipcode_98198,zipcode_98199
0,3,1.00,1180,5650,1.0,0,0,3,7,1180,...,0,0,0,0,0,0,1,0,0,0
1,3,2.25,2570,7242,2.0,0,0,3,7,2170,...,0,0,0,0,0,0,0,0,0,0
2,2,1.00,770,10000,1.0,0,0,3,6,770,...,0,0,0,0,0,0,0,0,0,0
3,4,3.00,1960,5000,1.0,0,0,5,7,1050,...,0,0,0,0,0,0,0,0,0,0
4,3,2.00,1680,8080,1.0,0,0,3,8,1680,...,0,0,0,0,0,0,0,0,0,0
5,4,4.50,5420,101930,1.0,0,0,3,11,3890,...,0,0,0,0,0,0,0,0,0,0
6,3,2.25,1715,6819,2.0,0,0,3,7,1715,...,0,0,0,0,0,0,0,0,0,0
7,3,1.50,1060,9711,1.0,0,0,3,7,1060,...,0,0,0,0,0,0,0,0,1,0
8,3,1.00,1780,7470,1.0,0,0,3,7,1050,...,1,0,0,0,0,0,0,0,0,0
9,3,2.50,1890,6560,2.0,0,0,3,7,1890,...,0,0,0,0,0,0,0,0,0,0


In [4]:
target_col = config["data"]["target_column"]
X = df.drop(columns=[target_col])
y = df[target_col]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=config["data"]["test_size"], 
    random_state=config["data"]["random_state"]
)

In [6]:
lin_cfg = config["linear_models"]
cv_strategy = KFold(
    n_splits=lin_cfg["cv_folds"],
    shuffle=True,
    random_state=config["data"]["random_state"]
)
alphas_grid = np.logspace(
    lin_cfg["alpha_grid"]["start_exp"],
    lin_cfg["alpha_grid"]["stop_exp"],
    lin_cfg["alpha_grid"]["num_points"]
)

In [7]:
models = {
    "Linear Regression" : Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=config["linear_models"]["alpha"], random_state=config["data"]["random_state"]))
    ]),
    
    "RidgeCV Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=alphas_grid, cv=cv_strategy))
    ]),
    
    "Lasso Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=config["linear_models"]["alpha"], max_iter=config["linear_models"]["max_iters"], random_state=config["data"]["random_state"]))
    ]),
     
    "LassoCV Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LassoCV(alphas=alphas_grid, cv=cv_strategy, max_iter=config["linear_models"]["max_iters"], random_state=config["data"]["random_state"]))
    ]),
    
    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=config["linear_models"]["alpha"], l1_ratio=config["linear_models"]["elastic_net"]["l1_ratio"], max_iter=config["linear_models"]["max_iters"], random_state=config["data"]["random_state"]))
    ]),
    
    "ElasticNetCV": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNetCV(alphas=alphas_grid, cv=cv_strategy, l1_ratio=config["linear_models"]["elastic_net"]["l1_ratios"], max_iter=config["linear_models"]["max_iters"], random_state=config["data"]["random_state"]))
    ]),
    
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(**config["distance_models"]["knn"]))
    ]),
    
    "Linear SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearSVR(dual="auto", 
                            C=config["distance_models"]["linear_svr"]["c_param"],
                            tol=config["distance_models"]["linear_svr"]["tol"],
                            max_iter=config["distance_models"]["linear_svr"]["max_iters"], 
                            random_state=config["data"]["random_state"]))
    ]),
    
    "Decision Tree": DecisionTreeRegressor(**config["tree_models"]["decision_tree"]),
    
    "Random Forest": RandomForestRegressor(**config["tree_models"]["random_forest"]),
    
    "Extra Trees": ExtraTreesRegressor(**config["tree_models"]["extra_trees"]),
    
    "Gradient Boosting": GradientBoostingRegressor(**config["boosting_models"]["gradient_boosting"]),
    
    "HistGradientBoosting": HistGradientBoostingRegressor(**config["boosting_models"]["hist_gradient_boosting"]),
    
    "XGBoost": XGBRegressor(**config["boosting_models"]["xgboost"]),
    
    "LightGBM": LGBMRegressor(**config["boosting_models"]["lightgbm"]),
    
    "CatBoost": CatBoostRegressor(**config["boosting_models"]["catboost"])
}

In [8]:
benchmark_results = []
trained_models = {}


y_test_dollars = np.exp(y_test)


print(f"Starting benchmark across {len(models)} machine learning algorithms...\n")
print("-" * 75)


for name, model in models.items():
    start_time = time.time()
    
    try:
        model.fit(X_train, y_train)
        fit_time = time.time() - start_time
        
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        y_pred_dollars = np.exp(y_test_pred)
        
        r2_train = r2_score(y_train, y_train_pred)
        r2_test = r2_score(y_test, y_test_pred)
            
        mae_dol = mean_absolute_error(y_test_dollars, y_pred_dollars)
        rmse_dol = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))
        mape = mean_absolute_percentage_error(y_test_dollars, y_pred_dollars) * 100
        
        benchmark_results.append({
            "Model": name,
            "Train R2 (Log)": round(r2_train, 4),
            "Test R2 (Log)": round(r2_test, 4),
            "Test MAE ($)": round(mae_dol, 2),
            "Test RMSE ($)": round(rmse_dol, 2),
            "Test MAPE (%)": round(mape, 2),
            "Train Time (s)": round(fit_time, 2)
        })
        trained_models[name] = model
        print(f"✓ {name:<22} | Test R²: {r2_test:.4f} | MAE: ${mae_dol:,.0f} | Time: {fit_time:.2f}s")
    except Exception as e:
        print(f"Failed {name}: {e}")
        

print("-" * 75)


Starting benchmark across 17 machine learning algorithms...

---------------------------------------------------------------------------
✓ Linear Regression      | Test R²: 0.8888 | MAE: $72,243 | Time: 0.18s
✓ Ridge Regression       | Test R²: 0.8886 | MAE: $72,336 | Time: 0.08s
✓ RidgeCV Regression     | Test R²: 0.8887 | MAE: $72,264 | Time: 4.23s
✓ Lasso Regression       | Test R²: -0.0000 | MAE: $226,304 | Time: 0.05s


d:\HousePricing\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.573796e+01, tolerance: 4.754e-01
  model = cd_fast.enet_coordinate_descent(


✓ LassoCV Regression     | Test R²: 0.8887 | MAE: $72,285 | Time: 77.59s
✓ ElasticNet             | Test R²: -0.0000 | MAE: $226,304 | Time: 0.04s
✓ ElasticNetCV           | Test R²: 0.8887 | MAE: $72,270 | Time: 111.81s
✓ K-Nearest Neighbors    | Test R²: 0.8501 | MAE: $85,065 | Time: 0.06s


d:\HousePricing\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


✓ Linear SVR             | Test R²: 0.8861 | MAE: $73,050 | Time: 27.02s
✓ Decision Tree          | Test R²: 0.8277 | MAE: $89,848 | Time: 0.29s
✓ Random Forest          | Test R²: 0.8899 | MAE: $71,376 | Time: 7.06s
✓ Extra Trees            | Test R²: 0.8926 | MAE: $70,377 | Time: 4.29s
✓ Gradient Boosting      | Test R²: 0.9032 | MAE: $66,286 | Time: 18.63s
✓ HistGradientBoosting   | Test R²: 0.9078 | MAE: $64,268 | Time: 1.98s
✓ XGBoost                | Test R²: 0.9082 | MAE: $64,161 | Time: 1.62s
✓ LightGBM               | Test R²: 0.9103 | MAE: $62,970 | Time: 0.83s
✓ CatBoost               | Test R²: 0.9088 | MAE: $64,132 | Time: 2.14s
---------------------------------------------------------------------------


In [9]:
results_df = pd.DataFrame(benchmark_results).sort_values(by="Test RMSE ($)", ascending=True).reset_index(drop=True)
display(results_df.style
        .highlight_min(subset=["Test MAE ($)", "Test RMSE ($)", "Test MAPE (%)", "Train Time (s)"], color="#d4edda")
        .highlight_max(subset=["Test R2 (Log)"], color="#d4edda")
        .format({"Test MAE ($)": "${:,.2f}", "Test RMSE ($)": "${:,.2f}", "Test MAPE (%)": "{:.2f}%", "Train Time (s)": "{:.2f}s"}))

,Model,Train R2 (Log),Test R2 (Log),Test MAE ($),Test RMSE ($),Test MAPE (%),Train Time (s)
0,LightGBM,0.933600,0.910300,"$62,970.10","$111,843.27",11.66%,0.83s
1,CatBoost,0.918000,0.908800,"$64,131.97","$112,645.09",11.87%,2.14s
2,XGBoost,0.938700,0.908200,"$64,161.41","$116,059.12",11.80%,1.62s
3,HistGradientBoosting,0.930100,0.907800,"$64,267.64","$116,462.56",11.80%,1.98s
4,Gradient Boosting,0.927900,0.903200,"$66,285.52","$118,188.58",12.16%,18.63s
5,Extra Trees,0.974500,0.892600,"$70,377.04","$128,698.87",12.75%,4.29s
6,Linear Regression,0.888400,0.888800,"$72,243.29","$130,544.24",13.30%,0.18s
7,RidgeCV Regression,0.888400,0.888700,"$72,264.21","$130,676.22",13.31%,4.23s
8,ElasticNetCV,0.888400,0.888700,"$72,269.56","$130,712.94",13.31%,111.81s
9,LassoCV Regression,0.888400,0.888700,"$72,285.12","$130,820.18",13.31%,77.59s
